In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split

nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\moohd\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
df = pd.read_csv('../cnn_dailymail/train.csv', nrows=10000) 

def prepare_data(df):
    X = []
    y = []
    
    for i, row in df.iterrows():
        article_sentences = sent_tokenize(row['article'])
        highlights = row['highlights']
        
        for sentence in article_sentences:
           
            label = 1 if any(word.lower() in highlights.lower() for word in word_tokenize(sentence) if len(word) > 3) else 0
            
            X.append(sentence)
            y.append(label)
            
    return X, y


sentences, labels = prepare_data(df)


vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_vectors = vectorizer.fit_transform(sentences)


X_train, X_test, y_train, y_test = train_test_split(X_vectors, labels, test_size=0.2, random_state=42)
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

print(f" Accuracy: {nb_model.score(X_test, y_test)}")


def summarize_article(article, model, vectorizer, num_sentences=3):
    
    source_sentences = sent_tokenize(article)
    if not source_sentences:
        return ""
    
    
    sentence_vectors = vectorizer.transform(source_sentences)
    
    
    probs = model.predict_proba(sentence_vectors)[:, 1]
    
    
    ranked_indices = np.argsort(probs)[::-1]
    
    
    top_indices = sorted(ranked_indices[:num_sentences])
    summary = " ".join([source_sentences[i] for i in top_indices])
    
    return summary


test_article = df['article'].iloc[10]
actual_highlight = df['highlights'].iloc[10]

generated_summary = summarize_article(test_article, nb_model, vectorizer)

print("\n------- Original Article Snippet ---")
print(test_article[:300] + "...")
print("\n--- Actual Highlight -------")
print(actual_highlight)
print("\n--- ---Naive Bayes Predicted Summary ---")
print(generated_summary)

 Accuracy: 0.6935707856499935

------- Original Article Snippet ---
By . Ellie Zolfagharifard . Take a look at a map today, and you’re likely to see that North America is larger than Africa, Alaska is larger than Mexico and China is smaller than Greenland. But in reality China is four times bigger than Greenland, Africa is three times bigger than North America and M...

--- Actual Highlight -------
The distortion is the result of the Mercator map which was created in 1596 to help sailors navigate the world .
It gives the right shapes of countries but at the cost of distorting sizes in favour of the wealthy lands to the north .
For instance, north America looks larger, or at least as big, as Africa, and Greenland also looks of comparable size .
In reality, you can fit north America into Africa and still have space for India, Argentina, Tunisia and some left over .
Map suggests Scandinavian countries are larger than India, whereas in reality India is three times the size .
The biggest ch